In [3]:
import numpy as np 
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns   

In [6]:
from pathlib import Path
import re
import sys

# Candidate paths to locate the dataset (workspace and common locations)
candidate_paths = [
    Path.cwd() / 'data' / 'pakwheels_cars_processed.csv',
    Path('f:/AI-DataScience/ANNProject/DataMining/data/pakwheels_cars_processed.csv'),
    Path.home() / 'data' / 'pakwheels_cars_processed.csv',
]

data_path = next((p for p in candidate_paths if p.exists()), None)
if data_path is None:
    raise FileNotFoundError(
        f"Could not find pakwheels_cars_processed.csv. Checked: {[str(p) for p in candidate_paths]}"
    )

df = pd.read_csv(data_path)

# --- Feature engineering for demo ---
# Extract year from `title`
def extract_year(title):
    if pd.isna(title):
        return np.nan
    m = re.search(r'\b(19|20)\d{2}\b', str(title))
    return int(m.group(0)) if m else np.nan

df['year'] = df.get('title', pd.Series()).apply(extract_year)

# Parse `price_raw` like 'PKR 25 lacs' or 'PKR 1.16 crore' into integer PKR
def parse_price(s):
    if pd.isna(s):
        return np.nan
    s = str(s).lower()
    multiplier = 1
    if 'crore' in s:
        multiplier = 10000000
    elif 'lakh' in s or 'lac' in s or 'lacs' in s:
        multiplier = 100000
    num = re.search(r'[\d\.,]+', s)
    if not num:
        return np.nan
    val = num.group(0).replace(',', '')
    try:
        return float(val) * multiplier
    except:
        return np.nan

df['price_parsed'] = df.get('price_raw', pd.Series()).apply(parse_price)
if 'price' in df.columns:
    df['price_parsed'] = df['price_parsed'].fillna(df['price'])
df['price_parsed'] = pd.to_numeric(df['price_parsed'], errors='coerce')
df['price_log'] = np.log1p(df['price_parsed'])

# Feature count from `features` text
def compute_feature_count(x):
    if pd.isna(x):
        return 0
    if isinstance(x, str):
        parts = re.split(r'[;,|]\s*', x)
        parts = [p for p in parts if p.strip()]
        return len(parts)
    try:
        return int(x)
    except:
        return 0

df['feature_count'] = df.get('features', pd.Series()).apply(compute_feature_count)

# Ensure `brand_clean` exists (infer from title if missing)
if 'brand_clean' not in df.columns and 'title' in df.columns:
    df['brand_clean'] = df['title'].str.split().str[0].str.capitalize()

# Encode categorical columns (simple integer encoding for demo)
from sklearn.preprocessing import LabelEncoder
enc_cols = ['body_type','assembly','exterior_color','registered_city','brand_clean']
for c in enc_cols:
    if c in df.columns:
        df[c+'_encoded'] = df[c].fillna('Unknown').astype(str)
        le = LabelEncoder()
        df[c+'_encoded'] = le.fit_transform(df[c+'_encoded'])

# Fill/enforce enrich_status
df['enrich_status'] = df.get('enrich_status', pd.Series()).fillna('Pending')

display(df.head())
df.dtypes

,title,url,price_raw,body_type,assembly,exterior_color,registered_city,features,enrich_status,price,...,price_log,feature_count,body_type_encoded,assembly_encoded,exterior_color_encoded,registered_city_encoded,brand_clean,year,price_parsed,brand_clean_encoded
0,Honda N One 2014 Premium for Sale\n6.6/10,https://www.pakwheels.com/used-cars/honda-n-on...,PKR 25 lacs,Other,Other,Other,Other,NaN,Done,2500000.0,...,14.731802,0,0,0,0,0,Honda,2014,2500000.0,3
1,Toyota Tank 2019 for Sale\n7.1/10,https://www.pakwheels.com/used-cars/toyota-tan...,PKR 36 lacs,Other,Other,Other,Other,NaN,Pending,3600000.0,...,15.096445,0,0,0,0,0,Toyota,2019,3600000.0,10
2,KIA Sportage L 2025 HEV for Sale,https://www.pakwheels.com/used-cars/kia-sporta...,PKR 1.16 crore,Other,Other,Other,Other,NaN,Pending,11600000.0,...,16.266516,0,0,0,0,0,KIA,2025,11600000.0,5
3,Toyota Hilux 2017 Revo G 3.0 for Sale,https://www.pakwheels.com/used-cars/toyota-hil...,PKR 78.5 lacs,Other,Other,Other,Other,NaN,Pending,7850000.0,...,15.876024,0,0,0,0,0,Toyota,2017,7850000.0,10
4,Toyota Yaris Hatchback 2021 X for Sale,https://www.pakwheels.com/used-cars/toyota-yar...,PKR 46.95 lacs,Other,Other,Other,Other,NaN,Pending,4695000.0,...,15.362009,0,0,0,0,0,Toyota,2021,4695000.0,10


title                       object
url                         object
price_raw                   object
body_type                   object
assembly                    object
exterior_color              object
registered_city             object
features                   float64
enrich_status               object
price                      float64
brand                       object
price_log                  float64
feature_count                int64
body_type_encoded            int32
assembly_encoded             int32
exterior_color_encoded       int32
registered_city_encoded      int32
brand_clean                 object
year                         int64
price_parsed               float64
brand_clean_encoded          int32
dtype: object